# 🧪 POC — LLM Local no Google Colab (Ollama + Gemma 4 E2B)
## Rodando Gemma 4 E2B via Ollama, sem API externa!

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/poc_ollama_gemma4_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivo:** Rodar um modelo de IA local no Google Colab gratuito (T4 GPU) usando **Ollama**,
sem precisar de API key externa.

**Modelo:** `gemma4:e2b` (~7.2 GB) — bom equilíbrio entre qualidade e velocidade.

**Por que Ollama?**
- API REST HTTP compatível com OpenAI em `localhost:11434`
- Alunos usam `requests.post()` — mesmo padrão da Gemini API
- Setup simples: `apt install` + `ollama pull` — sem compilação
- **Thinking mode** e **tool calling** nativos

**Status:** POC experimental — não é material de aula (ainda).

---


## 1. Verificar GPU

⚠️ **Importante:** Vá em `Runtime → Change runtime type` e selecione **T4 GPU** antes de continuar.


In [ ]:
# Verificar GPU disponível
!nvidia-smi


## 2. Instalar o Ollama

O Colab não tem `zstd` (necessário pro instalador) nem `lspci`/`lshw` (necessário pra detectar GPU).
Vamos instalar tudo e ajustar o `LD_LIBRARY_PATH` para a GPU funcionar.


In [ ]:
# Instalar dependências + Ollama
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

# Workaround: Ollama não detecta GPU no Colab sem isso
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

print("✅ Ollama instalado!")

## 3. Iniciar o Servidor Ollama

O Ollama roda como servidor em background na porta 11434.


In [ ]:
import subprocess, time, requests, os

# Matar instância anterior (seguro rodar múltiplas vezes)
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

# OLLAMA_KEEP_ALIVE=-1 mantém o modelo carregado indefinidamente na VRAM
# Sem isso, o Ollama descarrega após 5 min de inatividade (warm up de novo!)
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# Iniciar servidor em background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ}
)

# Esperar ficar pronto
print("⏳ Aguardando Ollama iniciar...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama rodando na porta 11434!")
            break
    except:
        time.sleep(1)
else:
    print("❌ Ollama não iniciou em 30s. Tente rodar a célula novamente.")

## 4. Baixar o Modelo

O **Gemma 4 E2B** tem ~7.2 GB de download. Leva ~2-5 min.


In [ ]:
# Baixar o modelo (Gemma 4 E2B — ~7.2 GB)
!ollama pull gemma4:e2b

print("\n✅ Modelo gemma4:e2b baixado!")

## 5. Teste Rápido

Verificar que o modelo está disponível e fazer um teste simples.


## 5. Carregar o Modelo (Warm Up)

⚠️ **A primeira inferência demora 30-60 segundos** (o modelo precisa ser carregado na GPU).
Depois disso, as respostas são rápidas (~10-20 tokens/s).

Para evitar warm up repetido, configuramos `OLLAMA_KEEP_ALIVE=-1` — isso mantém
o modelo na VRAM indefinidamente. Sem isso, o Ollama descarrega após 5 min de inatividade.


In [ ]:
# Listar modelos instalados
!ollama list

# Warm up: carregar o modelo na GPU (primeira inferência é lenta)
# Com OLLAMA_KEEP_ALIVE=-1, o modelo fica carregado e requests seguintes são rápidos
import time

print("🔥 Aquecendo o modelo (warm up) — a primeira inferência demora 30-60s...")
warmup_start = time.time()
warmup = requests.post("http://localhost:11434/api/chat", json={
    "model": "gemma4:e2b",
    "messages": [{"role": "user", "content": "Oi"}],
    "stream": False,
    "keep_alive": -1
}, timeout=240)
warmup_elapsed = time.time() - warmup_start
print(f"✅ Modelo carregado em {warmup_elapsed:.1f}s!")
print(f"   Resposta: {warmup.json()['message']['content'][:80]}")

# Agora as requests devem ser rápidas
print("\n--- Teste rápido via API (modelo já carregado) ---")
r = requests.post("http://localhost:11434/api/chat", json={
    "model": "gemma4:e2b",
    "messages": [{"role": "user", "content": "Diga: Olá, Guilda de IA!"}],
    "stream": False,
    "keep_alive": -1
}, timeout=120)
print(f"🤖 {r.json()['message']['content']}")

## 6. Inferência via API REST (`requests.post`)

O Ollama expõe uma API HTTP em `localhost:11434`.
Esse é o **ponto principal da POC**: o padrão é o mesmo da Gemini API — só muda a URL.


In [ ]:
import json

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "gemma4:e2b"

def chat(messages, model=None, stream=False):
    """Envia mensagens pro Ollama e retorna a resposta.

    Args:
        messages: lista [{"role": "...", "content": "..."}]
        model: nome do modelo (padrão: gemma4:e2b)
        stream: se True, retorna response object para iterar

    Returns:
        dict com a resposta (ou response se stream=True)
    """
    if model is None:
        model = MODEL
    # keep_alive=-1 mantém o modelo carregado indefinidamente (sem warm up entre requests)
    payload = {"model": model, "messages": messages, "stream": stream, "keep_alive": -1}
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    if response.status_code != 200:
        raise Exception(f"Erro {response.status_code}: {response.text}")
    return response if stream else response.json()

# Teste
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
])
print(f"🤖 {resultado['message']['content']}")
print(f"📊 Tokens: prompt={resultado.get('prompt_eval_count', '?')}, "
      f"completion={resultado.get('eval_count', '?')}")

## 7. Chat com Memória (Multi-turno)

Cada chamada envia o histórico completo. O modelo mantém o contexto.


In [ ]:
historico = [
    {"role": "system", "content": "Você é um assistente útil. Responda em português. Seja conciso."}
]

historico.append({"role": "user", "content": "Meu nome é Lucas e eu ensino IA."})
r1 = chat(historico)
historico.append({"role": "assistant", "content": r1["message"]["content"]})
print(f"🤖: {r1['message']['content']}")

historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico)
historico.append({"role": "assistant", "content": r2["message"]["content"]})
print(f"🤖: {r2['message']['content']}")

## 8. Modo Thinking (Raciocínio Interno)

O Gemma 4 suporta **thinking mode** — o modelo raciocina antes de responder.
Ative incluindo `/think` na mensagem.


In [ ]:
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Quantos números primos existem entre 1 e 20? /think"}
])

msg = resultado["message"]
thinking = msg.get("thinking", "(sem thinking)")
content = msg.get("content", "")
print(f"💭 Thinking:\n{thinking[:500]}...")
print(f"\n🤖 Resposta: {content}")

## 9. Streaming — Resposta em Tempo Real

Mostra o texto conforme é gerado, igual ao ChatGPT.


In [ ]:
print("🤖 ", end="")
response = chat(
    [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Explique o que é uma API em 3 frases."}
    ],
    stream=True
)
for line in response.iter_lines():
    if line:
        chunk = json.loads(line)
        if "message" in chunk and "content" in chunk["message"]:
            print(chunk["message"]["content"], end="", flush=True)
print()

## 10. API Compatível com OpenAI

O Ollama também tem `/v1/chat/completions` — formato **idêntico** à OpenAI.
Qualquer código OpenAI SDK pode apontar pro Ollama local trocando só a `base_url`.


In [ ]:
OPENAI_URL = "http://localhost:11434/v1/chat/completions"

payload = {
    "model": "gemma4:e2b",
    "messages": [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "O que é Python em uma frase?"}
    ],
    "temperature": 0.7,
    "max_tokens": 64,
    "keep_alive": -1
}

response = requests.post(OPENAI_URL, json=payload, timeout=300)  # 5min timeout para warm up
data = response.json()
print(f"🤖 {data['choices'][0]['message']['content']}")
print(f"📊 Model: {data['model']}, Usage: {data['usage']}")

## 11. Benchmark de Velocidade

Medindo tokens por segundo no Colab gratuito.


In [ ]:
import time

start = time.time()
resultado = chat([{"role": "user", "content": "Conte uma história curta sobre um robô que aprende a cozinhar."}])
elapsed = time.time() - start

eval_count = resultado.get("eval_count", 0)
eval_duration = resultado.get("eval_duration", 0) / 1e9
speed = eval_count / eval_duration if eval_duration > 0 else 0

print(f"⏱️ Tempo: {elapsed:.1f}s")
print(f"📊 Tokens: {resultado.get('prompt_eval_count', '?')} prompt, {eval_count} completion")
print(f"🚀 Velocidade: {speed:.1f} tokens/s")
print(f"\n📝 {resultado['message']['content'][:200]}...")

## 12. 📋 Resultados

### Warm Up ⏱️

A **primeira inferência demora ~2-3 minutos** (o modelo precisa ser carregado na GPU).
Depois disso, responses são rápidas: **~70 tokens/s**.

Para evitar warm up repetido, usamos `OLLAMA_KEEP_ALIVE=-1` — o modelo fica
na VRAM indefinidamente. Sem isso, o Ollama descarrega após 5 min de inatividade.

### O que funciona ✅

- **Gemma 4 E2B roda no Colab gratuito** (T4 GPU, 16 GB VRAM)
- Download: ~7.2 GB, VRAM: ~5-6 GB — sobra espaço
- Setup sem compilação: `apt install` + `ollama pull` (~5-7 min total)
- **Velocidade: ~70 tok/s** após warm up
- **API REST HTTP** — `requests.post()`, mesmo padrão da Gemini API
- **OpenAI-compatible** (`/v1/chat/completions`) — migração fácil
- **Thinking mode** (`/think`) e **tool calling** nativos
- Streaming funciona

### Limitações ⚠️

- **Warm up lento:** primeira inferência demora ~2-3 min (modelo carrega na GPU)
- **Download grande:** ~7.2 GB (demora ~2-5 min dependendo da conexão)
- **Sem persistência:** modelo baixado a cada sessão
- **Rate limits do Colab:** sessões gratuitas têm limite de horas
- **Workarounds:** `zstd`/`lspci`/`lshw` + `LD_LIBRARY_PATH` + `OLLAMA_KEEP_ALIVE=-1`

### Comparação com Gemini API

| Aspecto | Gemini API | Gemma 4 E2B Local |
|---------|-----------|-------------------|
| Setup | API key só | ~5-7 min instalação |
| Warm up | Instantâneo | **~2-3 min** (primeira req) |
| Velocidade | ~100+ tok/s | **~70 tok/s** (pós warm up) |
| Custos | Grátis com limites | 100% grátis |
| Privacidade | Dados vão pro Google | Fica no Colab |
| Qualidade | Muito alta | Boa (E2B) |

### Vabilidade para a Guilda

- **Aula 04 (Python + API):** Ollama como backend local — `requests.post()` sem API key
- **Aula futura (Agentes):** Tool calling nativo permite demonstrar agentes
- **Recomendação:** Complementar a Gemini API, mostrando o padrão HTTP unificado

---
*POC experimental — Guilda de IA UFVJM 2026.1*
